In [ ]:
import sys
import os

os.environ["CUDA_VISIBLE_DEVICES"]="0"

# 必须包含 ops 目录，因为 selective_scan_interface 通常在那
paths = [
    "/home/xiaolei/projects/baseline/ReID/CLIMB-ReID/mamba",
    "/home/xiaolei/projects/baseline/ReID/CLIMB-ReID/mamba/mamba_ssm/ops"
]

for p in paths:
    if p not in sys.path:
        sys.path.insert(0, p)

# 打印出来确认一下
print("\n".join(sys.path[:3]))

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import argparse
import glob

# 导入CLIMB模型
from climb.model import make_model, CLIMB
# 导入CAJR相关函数
from utils.caj_rerank import compute_jaccard_distance
from utils.metrics import euclidean_distance, org_cosine_similarity

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
class CustomImageDataset(Dataset):
    """自定义数据集，用于加载testData文件夹下的图像"""

    def __init__(self, image_dir, transform=None, selected_labels=None):
        """
        Args:
            image_dir: 图像文件夹路径
            transform: 图像变换
            selected_labels: 只提取指定的标签列表，如果为None则提取所有标签
        """
        self.image_dir = image_dir
        self.transform = transform
        self.image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
        self.image_paths.extend(glob.glob(os.path.join(image_dir, '*.png')))
        self.image_paths.sort()

        # 从文件名中提取标签信息
        # 格式如: b2859m9242_0000000_5114_01.jpg
        # 我们使用0000000（第二个下划线和第三个下划线之间的部分）作为类别标签
        self.labels = []
        for img_path in self.image_paths:
            filename = os.path.basename(img_path)
            # 提取类别标签部分（第二个下划线和第三个下划线之间的数字）
            parts = filename.split('_')
            if len(parts) >= 3:
                label = parts[1]  # 例如: 0000000
            else:
                label = 'unknown'
            self.labels.append(label)

        # 如果指定了selected_labels，只保留这些标签的数据
        if selected_labels is not None:
            selected_labels_set = set(selected_labels)
            # 过滤图像路径和标签
            filtered_indices = [i for i, label in enumerate(self.labels) if label in selected_labels_set]
            self.image_paths = [self.image_paths[i] for i in filtered_indices]
            self.labels = [self.labels[i] for i in filtered_indices]
            print(f"Filtered to selected labels: {selected_labels}")
            print(f"Kept {len(self.image_paths)} images")

        # 创建标签到数字的映射
        # 将标签转换为数字并排序
        unique_labels = sorted(list(set(self.labels)), key=lambda x: int(x))
        print(f"Unique labels (sorted by numeric value): {unique_labels}")

        self.label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}

        print(f"Found {len(self.image_paths)} images, {len(unique_labels)} unique IDs")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        from PIL import Image
        image = Image.open(img_path).convert('RGB')
        label_str = self.labels[idx]
        label = self.label_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label, label_str, img_path


def load_config(config_path):
    """加载配置文件"""
    from config.defaults import _C

    # 克隆默认配置
    cfg = _C.clone()

    # 使用CfgNode的merge_from_file方法合并配置
    cfg.merge_from_file(config_path)

    return cfg


def extract_features(model, dataloader, device):
    """
    使用CLIMB模型提取特征

    Returns:
        features_concat: torch tensor of shape (N, 1280) - 拼接特征
        features_feat1: torch tensor of shape (N, 768) - 基础特征
        features_feat2: torch tensor of shape (N, 512) - Spatial Mamba特征
        labels: numpy array of shape (N,)
        label_names: list of label names
    """
    model.eval()
    features_concat_list = []
    features_feat1_list = []
    features_feat2_list = []
    labels_list = []
    label_names_list = []

    with torch.no_grad():
        for images, labels, label_names, _ in tqdm(dataloader, desc='Extracting features'):
            images = images.to(device)

            # 获取特征 - 使用CLIMB模型的前向传播
            # 返回三个特征: feat_concat(拼接特征), feat1(基础特征), feat2(spatial mamba特征)
            feat_concat, feat1, feat2 = model(images)

            # 分别保存三种特征
            features_concat_list.append(feat_concat.cpu())
            features_feat1_list.append(feat1.cpu())
            features_feat2_list.append(feat2.cpu())
            labels_list.append(labels.numpy())
            label_names_list.extend(label_names)

    features_concat = torch.cat(features_concat_list, dim=0)
    features_feat1 = torch.cat(features_feat1_list, dim=0)
    features_feat2 = torch.cat(features_feat2_list, dim=0)
    labels = np.concatenate(labels_list, axis=0)

    return features_concat, features_feat1, features_feat2, labels, label_names_list


def apply_cajr_reranking(features, cam_labels=None, epoch='test'):
    """
    应用CAJR重排序后处理
    参考utils/metrics.py中的逻辑：
    1. 计算compute_jaccard_distance得到distmat_all
    2. 计算euclidean_distance得到distmat2
    3. 最终距离distmat = 0.5 * distmat1 + 0.5 * distmat2
    4. 使用距离矩阵进行特征聚合

    Args:
        features: torch tensor of shape (N, D) - 原始特征
        cam_labels: numpy array of shape (N,) - 相机标签（可选）
        epoch: str - epoch名称，用于日志输出

    Returns:
        refined_features: torch tensor of shape (N, D) - 经过CAJR重排序后的特征
    """
    print('=> Enter CAJR reranking')

    # 如果没有提供相机标签，使用全0
    if cam_labels is None:
        cam_labels = np.zeros(features.shape[0])
    elif isinstance(cam_labels, torch.Tensor):
        cam_labels = cam_labels.cpu().numpy()

    # 将特征转移到CPU并转为numpy
    if isinstance(features, torch.Tensor):
        features_np = features.cpu().numpy()
    else:
        features_np = features

    # 计算CAJR Jaccard距离矩阵（所有样本之间的距离）
    distmat_all = compute_jaccard_distance(
        features=features_np,
        cam_labels=cam_labels,
        epoch=epoch
    )

    # 将特征转为torch tensor以计算欧氏距离
    features_torch = torch.from_numpy(features_np).float()

    # 计算欧氏距离矩阵（所有样本之间的距离）
    distmat2 = euclidean_distance(features_torch, features_torch)

    # 最终距离矩阵：0.5 * Jaccard距离 + 0.5 * 欧氏距离
    distmat = 0.5 * distmat_all + 0.5 * distmat2

    print(f'Distance matrix shape: {distmat.shape}')

    # 使用距离矩阵进行特征聚合
    # 将距离转为相似度（负距离作为相似度）
    dist_tensor = torch.from_numpy(distmat).float()
    similarity = -dist_tensor

    # 对每个样本，使用softmax对相似度进行归一化
    weights = F.softmax(similarity, dim=1)

    # 计算加权特征
    refined_features = torch.mm(weights, features_torch)

    print(f'CAJR reranking completed. Refined features shape: {refined_features.shape}')

    return refined_features


def apply_tsne(features, perplexity=30, n_iter=1000, random_state=42):
    """
    应用t-SNE降维

    Args:
        features: numpy array of shape (N, feature_dim)
        perplexity: t-SNE perplexity参数
        n_iter: t-SNE迭代次数
        random_state: 随机种子

    Returns:
        tsne_features: numpy array of shape (N, 2)
    """
    print(f"Applying t-SNE with perplexity={perplexity}, n_iter={n_iter}...")
    tsne = TSNE(n_components=2, perplexity=perplexity, n_iter=n_iter,
                random_state=random_state, verbose=1)
    tsne_features = tsne.fit_transform(features)
    return tsne_features


def plot_tsne(tsne_features, labels, label_names, output_path='tsne_visualization.png',
              title='t-SNE Visualization'):
    """
    绘制t-SNE可视化结果

    Args:
        tsne_features: numpy array of shape (N, 2)
        labels: numpy array of shape (N,)
        label_names: list of label names
        output_path: 保存路径
        title: 图像标题
    """
    plt.figure(figsize=(16, 12))

    # 使用seaborn创建散点图
    unique_labels = np.unique(labels)

    # 定义五个固定颜色（RGB值）
    # 红色、蓝色、绿色、橙色、紫色
    fixed_colors = {
        '0140439': (1.0, 0.0, 0.0),    # 红色
        '0140449': (0.0, 0.0, 1.0),    # 蓝色
        '0140445': (0.0, 1.0, 0.0),    # 绿色
        '0140440': (1.0, 0.5, 0.0),    # 橙色
        '0140475': (0.5, 0.0, 0.5),     # 紫色
    }

    # 创建标签到颜色的映射
    label_to_color = {}
    for label in unique_labels:
        label_name = label_names[labels.tolist().index(label)]
        if label_name in fixed_colors:
            label_to_color[label] = fixed_colors[label_name]
        else:
            # 如果标签不在预定义颜色中，使用默认颜色
            label_to_color[label] = (0.5, 0.5, 0.5)

    for idx, label in enumerate(unique_labels):
        mask = labels == label
        plt.scatter(tsne_features[mask, 0], tsne_features[mask, 1],
                   c=[label_to_color[label]], label=label_names[labels.tolist().index(label)],
                   alpha=0.6, s=50, edgecolors='black', linewidth=0.3)

    # 移除坐标轴标注和标尺
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)

    # 只保留图例
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    plt.tight_layout()

    # 保存图像
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Visualization saved to {output_path}")
    plt.close()


def main(args):
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 加载配置文件
    cfg = load_config(args.config)
    print(f"Loaded config from {args.config}")

    # 创建数据变换（使用验证集的变换，不包含数据增强）
    val_transforms = T.Compose([
        T.Resize((cfg.INPUT.SIZE_TEST[0], cfg.INPUT.SIZE_TEST[1])),
        T.ToTensor(),
        T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD)
    ])

    # 只提取指定的标签
    selected_labels = ['0140439', '0140449', '0140445', '0140440', '0140475']

    # 创建数据集和数据加载器
    dataset = CustomImageDataset(args.image_dir, transform=val_transforms, selected_labels=selected_labels)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False,
                           num_workers=args.num_workers)

    # 创建模型
    print("Creating model...")
    num_classes = 751  # 使用默认类别数
    camera_num = 6
    view_num = 1

    model = make_model(cfg, num_classes, camera_num, view_num)

    # 加载预训练权重
    if args.pretrained_path:
        print(f"Loading pretrained weights from {args.pretrained_path}")
        # 尝试加载checkpoint
        try:
            checkpoint = torch.load(args.pretrained_path, map_location='cpu')
            if 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                state_dict = checkpoint

            # 移除'module.'前缀（如果是多GPU训练的模型）
            new_state_dict = {}
            for k, v in state_dict.items():
                name = k.replace('module.', '')
                new_state_dict[name] = v

            # 获取当前模型的参数名称
            model_state_dict = model.state_dict()

            # 只加载形状匹配的参数（忽略分类器和cv_embed等可能不匹配的层）
            matched_state_dict = {}
            mismatched_keys = []

            for name, param in new_state_dict.items():
                if name in model_state_dict:
                    if param.shape == model_state_dict[name].shape:
                        matched_state_dict[name] = param
                    else:
                        mismatched_keys.append(f"{name}: checkpoint shape {param.shape} vs model shape {model_state_dict[name].shape}")

            # 加载匹配的参数
            model.load_state_dict(matched_state_dict, strict=False)

            print(f"Loaded {len(matched_state_dict)} parameters successfully")
            if mismatched_keys:
                print("Ignored mismatched parameters:")
                for key in mismatched_keys:
                    print(f"  - {key}")

            print("Pretrained weights loaded successfully")
        except Exception as e:
            print(f"Warning: Failed to load pretrained weights: {e}")
            print("Using random initialization")

    model = model.to(device)

    # 提取特征
    print("\nExtracting features...")
    features_concat, features_feat1, features_feat2, labels, label_names = extract_features(model, dataloader, device)
    print(f"Extracted features_concat shape: {features_concat.shape}")
    print(f"Extracted features_feat1 shape: {features_feat1.shape}")
    print(f"Extracted features_feat2 shape: {features_feat2.shape}")
    print(f"Features dtype: {features_concat.dtype}")
    print(f"Features_concat range: [{features_concat.min():.4f}, {features_concat.max():.4f}]")
    print(f"Features_feat1 range: [{features_feat1.min():.4f}, {features_feat1.max():.4f}]")
    print(f"Features_feat2 range: [{features_feat2.min():.4f}, {features_feat2.max():.4f}]")

    # 应用CAJR重排序后处理
    print("\n" + "="*60)
    print("Applying CAJR reranking post-processing...")
    print("="*60)

    # 对拼接特征应用CAJR重排序
    features_concat_cajr = apply_cajr_reranking(features_concat, cam_labels=None, epoch='test')

    # 对基础特征应用CAJR重排序
    features_feat1_cajr = apply_cajr_reranking(features_feat1, cam_labels=None, epoch='test')

    # 对Spatial Mamba特征应用CAJR重排序
    features_feat2_cajr = apply_cajr_reranking(features_feat2, cam_labels=None, epoch='test')

    print(f"CAJR refined features_concat shape: {features_concat_cajr.shape}")
    print(f"CAJR refined features_feat1 shape: {features_feat1_cajr.shape}")
    print(f"CAJR refined features_feat2 shape: {features_feat2_cajr.shape}")

    # 将torch tensor转为numpy数组以便t-SNE处理
    features_concat_cajr_np = features_concat_cajr.numpy()
    features_feat1_cajr_np = features_feat1_cajr.numpy()
    features_feat2_cajr_np = features_feat2_cajr.numpy()

    # 应用t-SNE - 对三种经过CAJR处理的特征分别进行降维
    print("\nApplying t-SNE for CAJR refined concatenated features...")
    tsne_concat = apply_tsne(features_concat_cajr_np, perplexity=args.perplexity,
                            n_iter=args.n_iter, random_state=args.random_state)
    print(f"t-SNE CAJR refined concatenated features shape: {tsne_concat.shape}")

    print("\nApplying t-SNE for CAJR refined feat1 (base features)...")
    tsne_feat1 = apply_tsne(features_feat1_cajr_np, perplexity=args.perplexity,
                           n_iter=args.n_iter, random_state=args.random_state)
    print(f"t-SNE CAJR refined feat1 shape: {tsne_feat1.shape}")

    print("\nApplying t-SNE for CAJR refined feat2 (spatial mamba features)...")
    tsne_feat2 = apply_tsne(features_feat2_cajr_np, perplexity=args.perplexity,
                           n_iter=args.n_iter, random_state=args.random_state)
    print(f"t-SNE CAJR refined feat2 shape: {tsne_feat2.shape}")

    # 绘制可视化结果 - 三种经过CAJR处理特征的可视化
    print("\nPlotting visualizations...")
    output_dir = os.path.dirname(args.output)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 生成三个可视化文件
    base_name = os.path.splitext(args.output)[0]

    # 1. 经过CAJR处理的拼接特征的t-SNE可视化
    plot_tsne(tsne_concat, labels, label_names,
              output_path=f"{base_name}_CAJR_concat.png",
              title='t-SNE Visualization - CAJR Refined Concatenated Features (feat1 + feat2, 1280-dim)')

    # 2. 经过CAJR处理的基础特征的t-SNE可视化
    plot_tsne(tsne_feat1, labels, label_names,
              output_path=f"{base_name}_CAJR_feat1.png",
              title='t-SNE Visualization - CAJR Refined Base Features (768-dim)')

    # 3. 经过CAJR处理的Spatial Mamba特征的t-SNE可视化
    plot_tsne(tsne_feat2, labels, label_names,
              output_path=f"{base_name}_CAJR_feat2.png",
              title='t-SNE Visualization - CAJR Refined Spatial Mamba Features (512-dim)')

    # 保存特征和t-SNE结果（可选）
    if args.save_features:
        save_dir = os.path.dirname(args.output)
        np.save(os.path.join(save_dir, 'features_concat_CAJR.npy'), features_concat_cajr_np)
        np.save(os.path.join(save_dir, 'features_feat1_CAJR.npy'), features_feat1_cajr_np)
        np.save(os.path.join(save_dir, 'features_feat2_CAJR.npy'), features_feat2_cajr_np)
        np.save(os.path.join(save_dir, 'tsne_concat_CAJR.npy'), tsne_concat)
        np.save(os.path.join(save_dir, 'tsne_feat1_CAJR.npy'), tsne_feat1)
        np.save(os.path.join(save_dir, 'tsne_feat2_CAJR.npy'), tsne_feat2)
        np.save(os.path.join(save_dir, 'labels.npy'), labels)
        np.save(os.path.join(save_dir, 'label_names.npy'), np.array(label_names))
        print(f"\nCAJR refined features and t-SNE results saved to {save_dir}")

    print("\nDone! CAJR reranking and t-SNE visualization completed successfully.")


if __name__ == '__main__':

    # 在执行任何 argparse 解析之前加上这一段
    if 'ipykernel_launcher' in sys.argv[0]:
        # 只保留脚本名，清空所有干扰参数
        sys.argv = [sys.argv[0]]

    parser = argparse.ArgumentParser(description='t-SNE visualization with CAJR reranking for CLIMB ReID features')
    parser.add_argument('--config', type=str,
                        default='config/climb-vit-qiuxiuocclusion.yml',
                        help='Path to config file')
    parser.add_argument('--image_dir', type=str,
                        default='./testData/ReID_visual_data/m9242',
                        help='Path to image directory')
    parser.add_argument('--pretrained_path', type=str,
                        default='./logs/qiuxiu_occlusion/best_model.pth.tar',
                        help='Path to pretrained model checkpoint')
    parser.add_argument('--output', type=str,
                        default='tsne_visualization_CAJR.png',
                        help='Output base path for visualization (will generate 3 images with suffixes _CAJR_concat, _CAJR_feat1, _CAJR_feat2)')
    parser.add_argument('--batch_size', type=int, default=64,
                        help='Batch size for feature extraction')
    parser.add_argument('--num_workers', type=int, default=4,
                        help='Number of workers for data loading')
    parser.add_argument('--perplexity', type=int, default=30,
                        help='t-SNE perplexity parameter')
    parser.add_argument('--n_iter', type=int, default=1000,
                        help='t-SNE number of iterations')
    parser.add_argument('--random_state', type=int, default=42,
                        help='Random seed for t-SNE')
    parser.add_argument('--save_features', action='store_true',
                        help='Save features and t-SNE results to .npy files')

    args = parser.parse_args()

    main(args)

In [ ]:
class CustomImageDataset(Dataset):
    """自定义数据集，用于加载testData文件夹下的图像"""

    def __init__(self, image_dir, transform=None):
        """
        Args:
            image_dir: 图像文件夹路径
            transform: 图像变换
        """
        self.image_dir = image_dir
        self.transform = transform
        self.image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
        self.image_paths.extend(glob.glob(os.path.join(image_dir, '*.png')))
        self.image_paths.sort()

        # 从文件名中提取标签信息
        # 格式如: b2859m9242_0000000_5114_01.jpg
        # 我们使用0000000（第二个下划线和第三个下划线之间的部分）作为类别标签
        self.labels = []
        for img_path in self.image_paths:
            filename = os.path.basename(img_path)
            # 提取类别标签部分（第二个下划线和第三个下划线之间的数字）
            parts = filename.split('_')
            if len(parts) >= 3:
                label = parts[1]  # 例如: 0000000
            else:
                label = 'unknown'
            self.labels.append(label)

        # 创建标签到数字的映射
        # 将标签转换为数字并排序
        unique_labels = sorted(list(set(self.labels)), key=lambda x: int(x))
        print(f"Unique labels (sorted by numeric value): {unique_labels}")

        self.label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}

        print(f"Found {len(self.image_paths)} images, {len(unique_labels)} unique IDs")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        from PIL import Image
        image = Image.open(img_path).convert('RGB')
        label_str = self.labels[idx]
        label = self.label_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label, label_str, img_path


def load_config(config_path):
    """加载配置文件"""
    from config.defaults import _C

    # 克隆默认配置
    cfg = _C.clone()

    # 使用CfgNode的merge_from_file方法合并配置
    cfg.merge_from_file(config_path)

    return cfg


def extract_features(model, dataloader, device):
    """
    使用CLIMB模型提取特征

    Returns:
        features_concat: torch tensor of shape (N, 1280) - 拼接特征
        features_feat1: torch tensor of shape (N, 768) - 基础特征
        features_feat2: torch tensor of shape (N, 512) - Spatial Mamba特征
        labels: numpy array of shape (N,)
        label_names: list of label names
    """
    model.eval()
    features_concat_list = []
    features_feat1_list = []
    features_feat2_list = []
    labels_list = []
    label_names_list = []

    with torch.no_grad():
        for images, labels, label_names, _ in tqdm(dataloader, desc='Extracting features'):
            images = images.to(device)

            # 获取特征 - 使用CLIMB模型的前向传播
            # 返回三个特征: feat_concat(拼接特征), feat1(基础特征), feat2(spatial mamba特征)
            feat_concat, feat1, feat2 = model(images)

            # 分别保存三种特征
            features_concat_list.append(feat_concat.cpu())
            features_feat1_list.append(feat1.cpu())
            features_feat2_list.append(feat2.cpu())
            labels_list.append(labels.numpy())
            label_names_list.extend(label_names)

    features_concat = torch.cat(features_concat_list, dim=0)
    features_feat1 = torch.cat(features_feat1_list, dim=0)
    features_feat2 = torch.cat(features_feat2_list, dim=0)
    labels = np.concatenate(labels_list, axis=0)

    return features_concat, features_feat1, features_feat2, labels, label_names_list


def apply_cajr_reranking(features, cam_labels=None, epoch='test'):
    """
    应用CAJR重排序后处理
    参考utils/metrics.py中的逻辑：
    1. 计算compute_jaccard_distance得到distmat_all
    2. 计算euclidean_distance得到distmat2
    3. 最终距离distmat = 0.5 * distmat1 + 0.5 * distmat2
    4. 使用距离矩阵进行特征聚合

    Args:
        features: torch tensor of shape (N, D) - 原始特征
        cam_labels: numpy array of shape (N,) - 相机标签（可选）
        epoch: str - epoch名称，用于日志输出

    Returns:
        refined_features: torch tensor of shape (N, D) - 经过CAJR重排序后的特征
    """
    print('=> Enter CAJR reranking')

    # 如果没有提供相机标签，使用全0
    if cam_labels is None:
        cam_labels = np.zeros(features.shape[0])
    elif isinstance(cam_labels, torch.Tensor):
        cam_labels = cam_labels.cpu().numpy()

    # 将特征转移到CPU并转为numpy
    if isinstance(features, torch.Tensor):
        features_np = features.cpu().numpy()
    else:
        features_np = features

    # 计算CAJR Jaccard距离矩阵（所有样本之间的距离）
    distmat_all = compute_jaccard_distance(
        features=features_np,
        cam_labels=cam_labels,
        epoch=epoch
    )

    # 将特征转为torch tensor以计算欧氏距离
    features_torch = torch.from_numpy(features_np).float()

    # 计算欧氏距离矩阵（所有样本之间的距离）
    distmat2 = euclidean_distance(features_torch, features_torch)

    # 最终距离矩阵：0.5 * Jaccard距离 + 0.5 * 欧氏距离
    distmat = 0.5 * distmat_all + 0.5 * distmat2

    print(f'Distance matrix shape: {distmat.shape}')

    # 使用距离矩阵进行特征聚合
    # 将距离转为相似度（负距离作为相似度）
    dist_tensor = torch.from_numpy(distmat).float()
    similarity = -dist_tensor

    # 对每个样本，使用softmax对相似度进行归一化
    weights = F.softmax(similarity, dim=1)

    # 计算加权特征
    refined_features = torch.mm(weights, features_torch)

    print(f'CAJR reranking completed. Refined features shape: {refined_features.shape}')

    return refined_features


def apply_tsne(features, perplexity=30, n_iter=1000, random_state=42):
    """
    应用t-SNE降维

    Args:
        features: numpy array of shape (N, feature_dim)
        perplexity: t-SNE perplexity参数
        n_iter: t-SNE迭代次数
        random_state: 随机种子

    Returns:
        tsne_features: numpy array of shape (N, 2)
    """
    print(f"Applying t-SNE with perplexity={perplexity}, n_iter={n_iter}...")
    tsne = TSNE(n_components=2, perplexity=perplexity, n_iter=n_iter,
                random_state=random_state, verbose=1)
    tsne_features = tsne.fit_transform(features)
    return tsne_features


def plot_tsne(tsne_features, labels, label_names, output_path='tsne_visualization.png',
              title='t-SNE Visualization'):
    """
    绘制t-SNE可视化结果

    Args:
        tsne_features: numpy array of shape (N, 2)
        labels: numpy array of shape (N,)
        label_names: list of label names
        output_path: 保存路径
        title: 图像标题
    """
    plt.figure(figsize=(16, 12))

    # 使用seaborn创建散点图
    unique_labels = np.unique(labels)

    # 创建20个不同的颜色（RGB值）
    color_palette = [
        (1.0, 0.0, 0.0),      # 红色
        (0.0, 0.0, 1.0),      # 蓝色
        (0.0, 1.0, 0.0),      # 绿色
        (1.0, 0.5, 0.0),      # 橙色
        (0.5, 0.0, 0.5),      # 紫色
        (0.0, 1.0, 1.0),      # 青色
        (1.0, 0.0, 1.0),      # 洋红
        (1.0, 1.0, 0.0),      # 黄色
        (0.5, 0.0, 0.0),      # 深红
        (0.0, 0.0, 0.5),      # 深蓝
        (0.0, 0.5, 0.0),      # 深绿
        (0.5, 0.25, 0.0),     # 深橙
        (0.25, 0.0, 0.5),     # 深紫
        (0.0, 0.5, 0.5),      # 深青
        (0.5, 0.0, 0.5),      # 深洋红
        (0.5, 0.5, 0.0),      # 深黄
        (1.0, 0.75, 0.8),     # 粉色
        (0.8, 0.8, 0.8),      # 灰色
        (0.0, 0.0, 0.0),      # 黑色
        (1.0, 0.4, 0.4)       # 浅红
    ]

    # 按类别顺序分配颜色
    label_to_color = {}
    for idx, label in enumerate(unique_labels):
        label_to_color[label] = color_palette[idx % len(color_palette)]

    for idx, label in enumerate(unique_labels):
        mask = labels == label
        plt.scatter(tsne_features[mask, 0], tsne_features[mask, 1],
                   c=[label_to_color[label]], label=label_names[labels.tolist().index(label)],
                   alpha=0.6, s=50, edgecolors='black', linewidth=0.3)

    # 移除坐标轴标注和标尺
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)

    # 将图例放在图片正下方，增大文字
    # plt.legend(bbox_to_anchor=(0.5, -0.05), loc='upper center', ncol=len(unique_labels), fontsize=12)
    plt.legend(
        bbox_to_anchor=(0, -0.08),  # x=0.0 表示画布最左侧，y=-0.06 保持在下方
        loc='lower left',              # 图例的“左下边缘”与锚点对齐
        ncol=7,
        fontsize=15
    )
    plt.tight_layout()

    # 保存图像
    plt.savefig(output_path, dpi=600, bbox_inches='tight')
    print(f"Visualization saved to {output_path}")
    plt.close()


def main(args):
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 加载配置文件
    cfg = load_config(args.config)
    print(f"Loaded config from {args.config}")

    # 创建数据变换（使用验证集的变换，不包含数据增强）
    val_transforms = T.Compose([
        T.Resize((cfg.INPUT.SIZE_TEST[0], cfg.INPUT.SIZE_TEST[1])),
        T.ToTensor(),
        T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD)
    ])

    # 创建数据集和数据加载器（加载所有类别的图像）
    dataset = CustomImageDataset(args.image_dir, transform=val_transforms)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False,
                           num_workers=args.num_workers)

    # 创建模型
    print("Creating model...")
    num_classes = 751  # 使用默认类别数
    camera_num = 6
    view_num = 1

    model = make_model(cfg, num_classes, camera_num, view_num)

    # 加载预训练权重
    if args.pretrained_path:
        print(f"Loading pretrained weights from {args.pretrained_path}")
        # 尝试加载checkpoint
        try:
            checkpoint = torch.load(args.pretrained_path, map_location='cpu')
            if 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                state_dict = checkpoint

            # 移除'module.'前缀（如果是多GPU训练的模型）
            new_state_dict = {}
            for k, v in state_dict.items():
                name = k.replace('module.', '')
                new_state_dict[name] = v

            # 获取当前模型的参数名称
            model_state_dict = model.state_dict()

            # 只加载形状匹配的参数（忽略分类器和cv_embed等可能不匹配的层）
            matched_state_dict = {}
            mismatched_keys = []

            for name, param in new_state_dict.items():
                if name in model_state_dict:
                    if param.shape == model_state_dict[name].shape:
                        matched_state_dict[name] = param
                    else:
                        mismatched_keys.append(f"{name}: checkpoint shape {param.shape} vs model shape {model_state_dict[name].shape}")

            # 加载匹配的参数
            model.load_state_dict(matched_state_dict, strict=False)

            print(f"Loaded {len(matched_state_dict)} parameters successfully")
            if mismatched_keys:
                print("Ignored mismatched parameters:")
                for key in mismatched_keys:
                    print(f"  - {key}")

            print("Pretrained weights loaded successfully")
        except Exception as e:
            print(f"Warning: Failed to load pretrained weights: {e}")
            print("Using random initialization")

    model = model.to(device)

    # 提取特征
    print("\nExtracting features...")
    features_concat, features_feat1, features_feat2, labels, label_names = extract_features(model, dataloader, device)
    print(f"Extracted features_concat shape: {features_concat.shape}")
    print(f"Extracted features_feat1 shape: {features_feat1.shape}")
    print(f"Extracted features_feat2 shape: {features_feat2.shape}")
    print(f"Features dtype: {features_concat.dtype}")
    print(f"Features_concat range: [{features_concat.min():.4f}, {features_concat.max():.4f}]")
    print(f"Features_feat1 range: [{features_feat1.min():.4f}, {features_feat1.max():.4f}]")
    print(f"Features_feat2 range: [{features_feat2.min():.4f}, {features_feat2.max():.4f}]")

    # 应用CAJR重排序后处理
    print("\n" + "="*60)
    print("Applying CAJR reranking post-processing...")
    print("="*60)

    # 对拼接特征应用CAJR重排序
    features_concat_cajr = apply_cajr_reranking(features_concat, cam_labels=None, epoch='test')

    # 对基础特征应用CAJR重排序
    features_feat1_cajr = apply_cajr_reranking(features_feat1, cam_labels=None, epoch='test')

    # 对Spatial Mamba特征应用CAJR重排序
    features_feat2_cajr = apply_cajr_reranking(features_feat2, cam_labels=None, epoch='test')

    print(f"CAJR refined features_concat shape: {features_concat_cajr.shape}")
    print(f"CAJR refined features_feat1 shape: {features_feat1_cajr.shape}")
    print(f"CAJR refined features_feat2 shape: {features_feat2_cajr.shape}")

    # 将torch tensor转为numpy数组以便t-SNE处理
    features_concat_cajr_np = features_concat_cajr.numpy()
    features_feat1_cajr_np = features_feat1_cajr.numpy()
    features_feat2_cajr_np = features_feat2_cajr.numpy()

    # 应用t-SNE - 对三种经过CAJR处理的特征分别进行降维
    print("\nApplying t-SNE for CAJR refined concatenated features...")
    tsne_concat = apply_tsne(features_concat_cajr_np, perplexity=args.perplexity,
                            n_iter=args.n_iter, random_state=args.random_state)
    print(f"t-SNE CAJR refined concatenated features shape: {tsne_concat.shape}")

    print("\nApplying t-SNE for CAJR refined feat1 (base features)...")
    tsne_feat1 = apply_tsne(features_feat1_cajr_np, perplexity=args.perplexity,
                           n_iter=args.n_iter, random_state=args.random_state)
    print(f"t-SNE CAJR refined feat1 shape: {tsne_feat1.shape}")

    print("\nApplying t-SNE for CAJR refined feat2 (spatial mamba features)...")
    tsne_feat2 = apply_tsne(features_feat2_cajr_np, perplexity=args.perplexity,
                           n_iter=args.n_iter, random_state=args.random_state)
    print(f"t-SNE CAJR refined feat2 shape: {tsne_feat2.shape}")

    # 绘制可视化结果 - 三种经过CAJR处理特征的可视化
    print("\nPlotting visualizations...")
    output_dir = os.path.dirname(args.output)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 生成三个可视化文件
    base_name = os.path.splitext(args.output)[0]

    # 1. 经过CAJR处理的拼接特征的t-SNE可视化
    plot_tsne(tsne_concat, labels, label_names,
              output_path=f"{base_name}_CAJR_concat.png",
              title='t-SNE Visualization - CAJR Refined Concatenated Features (feat1 + feat2, 1280-dim)')

    # 2. 经过CAJR处理的基础特征的t-SNE可视化
    plot_tsne(tsne_feat1, labels, label_names,
              output_path=f"{base_name}_CAJR_feat1.png",
              title='t-SNE Visualization - CAJR Refined Base Features (768-dim)')

    # 3. 经过CAJR处理的Spatial Mamba特征的t-SNE可视化
    plot_tsne(tsne_feat2, labels, label_names,
              output_path=f"{base_name}_CAJR_feat2.png",
              title='t-SNE Visualization - CAJR Refined Spatial Mamba Features (512-dim)')

    # 保存特征和t-SNE结果（可选）
    if args.save_features:
        save_dir = os.path.dirname(args.output)
        np.save(os.path.join(save_dir, 'features_concat_CAJR.npy'), features_concat_cajr_np)
        np.save(os.path.join(save_dir, 'features_feat1_CAJR.npy'), features_feat1_cajr_np)
        np.save(os.path.join(save_dir, 'features_feat2_CAJR.npy'), features_feat2_cajr_np)
        np.save(os.path.join(save_dir, 'tsne_concat_CAJR.npy'), tsne_concat)
        np.save(os.path.join(save_dir, 'tsne_feat1_CAJR.npy'), tsne_feat1)
        np.save(os.path.join(save_dir, 'tsne_feat2_CAJR.npy'), tsne_feat2)
        np.save(os.path.join(save_dir, 'labels.npy'), labels)
        np.save(os.path.join(save_dir, 'label_names.npy'), np.array(label_names))
        print(f"\nCAJR refined features and t-SNE results saved to {save_dir}")

    print("\nDone! CAJR reranking and t-SNE visualization completed successfully.")


if __name__ == '__main__':

    # 在执行任何 argparse 解析之前加上这一段
    if 'ipykernel_launcher' in sys.argv[0]:
        # 只保留脚本名，清空所有干扰参数
        sys.argv = [sys.argv[0]]

    parser = argparse.ArgumentParser(description='t-SNE visualization with CAJR reranking for CLIMB ReID features')
    parser.add_argument('--config', type=str,
                        default='config/climb-vit-qiuxiuocclusion.yml',
                        help='Path to config file')
    parser.add_argument('--image_dir', type=str,
                        default='./testData/ReID_visual_data/m9366',
                        help='Path to image directory')
    parser.add_argument('--pretrained_path', type=str,
                        default='./logs/qiuxiu_occlusion/best_model.pth.tar',
                        help='Path to pretrained model checkpoint')
    parser.add_argument('--output', type=str,
                        default='tsne_visualization_CAJR_9366.png',
                        help='Output base path for visualization (will generate 3 images with suffixes _CAJR_concat, _CAJR_feat1, _CAJR_feat2)')
    parser.add_argument('--batch_size', type=int, default=64,
                        help='Batch size for feature extraction')
    parser.add_argument('--num_workers', type=int, default=4,
                        help='Number of workers for data loading')
    parser.add_argument('--perplexity', type=int, default=30,
                        help='t-SNE perplexity parameter')
    parser.add_argument('--n_iter', type=int, default=1000,
                        help='t-SNE number of iterations')
    parser.add_argument('--random_state', type=int, default=42,
                        help='Random seed for t-SNE')
    parser.add_argument('--save_features', action='store_true',
                        help='Save features and t-SNE results to .npy files')

    args = parser.parse_args()

    main(args)